# Position-bias study with inverse propensity weighting

Issue: #32 (includes the weight column of the former #33)

## Question

Clicks depend on where an item was shown, not only on whether it is relevant. If we train a ranker on such clicks, it learns the old ranking's positions. Does weighting each click by `1 / propensity` (IPW) recover the true relevance ordering, and what does clipping the weights do?

## Data used

The simulated impressions of #31 (logging policy popularity + noise, position-based clicks with known propensity `theta_k = 1/k`, true relevance from the real held-out labels), regenerated here for several seeds; the features of #22 for the shown items. **Test:** 10,000 evaluation queries (200 candidates each) scored against **true relevance** (0.02, 0.5, 0.7, 0.9 for label 0 to 3), not against clicks.

Code: `notebooks/clicksim.py`, `notebooks/dcn.py` (weight column), `notebooks/check_weights_torch.py`.

**Models.** LightGBM with the binary objective on `click`, 18 features, **position is never a feature** (it is not known when ranking). Variants:
- *naive*: every row has weight 1;
- *IPW*: a clicked row gets weight `min(1 / theta_k, M)`, an unclicked row gets weight 1, then all weights are divided by their mean (**self-normalisation**, so the average weight is 1). `M` is the **clipping** threshold: M = 1 is the naive model, M = infinity is plain IPW.

Because LightGBM cannot share a process with PyTorch on macOS, the neural weight-column check runs as a separate script.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from clicksim import R_BY_LABEL, simulate_clicks
from features import FEATURE_COLUMNS

DATA = Path("data") if Path("data").exists() else Path("../data")
ROOT = DATA.parent
POOL = 200
NUM = [c for c in FEATURE_COLUMNS if c not in ("category", "category_conf")]
N_SEEDS = 8
CLIPS = [1, 2, 5, 10, 20, 1e9]          # 1 = naive (all weights 1), 1e9 = no clipping

## The weight column, checked

1. LightGBM: weights of 1 reproduce the unweighted model; non-uniform weights change it.
2. The neural loss (in a separate process): weights of 1 reproduce the unweighted loss and training; zero weights give zero loss; non-uniform weights change both.

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(2000, 5)).astype(np.float32)
y = (X[:, 0] + rng.normal(size=2000) > 0).astype(int)
params = {"objective": "binary", "verbose": -1, "seed": 0, "deterministic": True, "force_col_wise": True, "num_leaves": 7}
p_none = lgb.train(params, lgb.Dataset(X, y), 20).predict(X)
p_ones = lgb.train(params, lgb.Dataset(X, y, weight=np.ones(2000)), 20).predict(X)
p_w = lgb.train(params, lgb.Dataset(X, y, weight=np.linspace(0.1, 5, 2000)), 20).predict(X)
assert np.allclose(p_none, p_ones, atol=1e-9), "weights of 1 changed the LightGBM model"
assert not np.allclose(p_none, p_w, atol=1e-3), "non-uniform weights did not change the LightGBM model"
print("LightGBM: weights of 1 reproduce the unweighted model; non-uniform weights change it")

out = subprocess.run([sys.executable, str(ROOT / "notebooks" / "check_weights_torch.py")], capture_output=True, text=True, check=False)
assert out.returncode == 0, out.stderr[-1500:]
tres = json.loads([ln for ln in out.stdout.splitlines() if ln.startswith("RESULT ")][-1][7:])
assert abs(tres["loss_plain"] - tres["loss_ones"]) < 1e-6 and tres["loss_zero"] == 0.0
assert tres["loss_weighted"] != tres["loss_plain"] and tres["train_ones_equals_plain"] and tres["train_weighted_differs"]
print("neural loss:", tres)

## Load the data

In [ ]:
cols = ["session", "aid", "label", "log_clicks"] + NUM
tr = pl.read_parquet(DATA / "features" / "train_asof11.parquet", columns=list(dict.fromkeys(cols)))
n_q = tr.height // POOL
X_all = tr.select(NUM).to_numpy().astype(np.float32)          # row = query * 200 + candidate
labels = tr["label"].to_numpy().reshape(n_q, POOL).astype(np.int64)
popularity = tr["log_clicks"].to_numpy().reshape(n_q, POOL).astype(np.float64)
sess = tr["session"].to_numpy().reshape(n_q, POOL)[:, 0]
aid = tr["aid"].to_numpy().reshape(n_q, POOL)
del tr

ev = pl.read_parquet(DATA / "features" / "eval_w0.parquet", columns=list(dict.fromkeys(["label", "log_clicks"] + NUM))).head(10_000 * POOL)
X_test = ev.select(NUM).to_numpy().astype(np.float32)
gain_test = R_BY_LABEL[ev["label"].to_numpy().reshape(-1, POOL)]          # true relevance
pop_test = ev["log_clicks"].to_numpy().reshape(-1, POOL)
del ev
print(f"training queries {n_q:,}; test queries {len(gain_test):,}; features {len(NUM)}")

# the simulation here (seed 0) must equal the impressions saved in #31
s0 = simulate_clicks(popularity, labels, seed=0)
saved = pl.read_parquet(DATA / "sim" / "impressions_train.parquet")
assert (saved["click"].to_numpy() == s0["click"]).all() and (saved["position"].to_numpy() == s0["position"]).all()
print("seed 0 reproduces data/sim/impressions_train.parquet")

In [ ]:
sim_cfg = json.loads((DATA / "sim" / "config.json").read_text())
print("simulation settings:", {k: sim_cfg[k] for k in ("k", "eta", "sigma", "seed", "r_by_label", "propensity")})
print("propensities used (theta_k for positions 1-20):", [round(1 / k ** sim_cfg["eta"], 4) for k in range(1, sim_cfg["k"] + 1)])
print(f"this notebook re-simulates seeds 0 to {N_SEEDS - 1} with the same settings")

## Ground-truth evaluation

NDCG@10 of a ranking of the 200 candidates of a test query, with gain = true relevance and the ideal ordering from the true relevance.

In [ ]:
DISC = 1.0 / np.log2(np.arange(2, 12))


def ndcg_true(scores, gains):
    top = np.argsort(-scores, axis=1, kind="stable")[:, :10]
    disc = DISC[: top.shape[1]]
    dcg = (np.take_along_axis(gains, top, axis=1) * disc).sum(axis=1)
    ideal = (-np.sort(-gains, axis=1)[:, : top.shape[1]] * disc).sum(axis=1)
    return dcg / ideal


# toy check by hand: gains [0.9, 0.02, 0.5]. Best order is [0.9, 0.5, 0.02]; worst order is [0.02, 0.5, 0.9].
g = np.array([[0.9, 0.02, 0.5]])
perfect = ndcg_true(np.array([[3.0, 1.0, 2.0]]), g)[0]
worst = ndcg_true(np.array([[1.0, 3.0, 2.0]]), g)[0]
assert abs(perfect - 1.0) < 1e-12
expected_worst = (0.02 / np.log2(2) + 0.5 / np.log2(3) + 0.9 / np.log2(4)) / (0.9 / np.log2(2) + 0.5 / np.log2(3) + 0.02 / np.log2(4))
assert abs(worst - expected_worst) < 1e-12
print("ground-truth NDCG: perfect order", round(perfect, 3), " worst order", round(worst, 3))

rng = np.random.default_rng(0)
random_ndcg = ndcg_true(rng.random(gain_test.shape), gain_test).mean()
logging_ndcg = ndcg_true(pop_test, gain_test).mean()
print(f"reference: random order {random_ndcg:.4f}; the logging policy's popularity order {logging_ndcg:.4f}")

## Train naive and IPW models over several simulation seeds and clipping thresholds

In [ ]:
P = {"objective": "binary", "learning_rate": 0.05, "num_leaves": 15, "min_data_in_leaf": 100, "feature_fraction": 0.8,
     "bagging_fraction": 0.8, "bagging_freq": 1, "verbose": -1, "seed": 0, "deterministic": True, "force_col_wise": True}
ROUNDS = 150


def weights_for(click, propensity, clip):
    w = np.where(click == 1, np.minimum(1.0 / propensity, clip), 1.0)
    return w / w.mean()                       # self-normalisation


rows = []
t0 = time.time()
for seed in range(N_SEEDS):
    sim = simulate_clicks(popularity, labels, seed=seed)
    idx = sim["query"] * POOL + sim["candidate"]
    Xs, click, theta = X_all[idx], sim["click"], sim["propensity"].astype(np.float64)
    for clip in CLIPS:
        w = weights_for(click, theta, clip)
        model = lgb.train(P, lgb.Dataset(Xs, click, weight=w), ROUNDS)
        score = model.predict(X_test).reshape(-1, POOL)
        rows.append({"seed": seed, "clip": clip, "ndcg_true": float(ndcg_true(score, gain_test).mean()), "clicks": int(click.sum()),
                     "max_weight": float(w.max()), "weight_sd": float(w.std())})
    print(f"seed {seed} done ({time.time() - t0:.0f} s)")
res = pl.DataFrame(rows)

In [ ]:
# reference: a model trained on the real labels of all 200 candidates (about 10x more positives, so NOT a fair upper bound)
y_real = (labels.reshape(-1) > 0).astype(int)
ref_model = lgb.train(P, lgb.Dataset(X_all, y_real), ROUNDS)
ref_ndcg = float(ndcg_true(ref_model.predict(X_test).reshape(-1, POOL), gain_test).mean())
print(f"reference trained on the real labels of all candidates: {ref_ndcg:.4f}")

naive = res.filter(pl.col("clip") == 1).sort("seed")["ndcg_true"].to_numpy()
rows = []
for clip in CLIPS:
    part = res.filter(pl.col("clip") == clip).sort("seed")
    v = part["ndcg_true"].to_numpy()
    d = v - naive
    rows.append({"clip M": "1 (naive)" if clip == 1 else ("none" if clip >= 1e8 else str(int(clip))),
                 "mean NDCG@10 (true)": round(v.mean(), 4), "sd over seeds": round(v.std(ddof=1), 4),
                 "mean diff to naive": round(d.mean(), 4), "diff 95% CI": f"[{d.mean() - 2.365 * d.std(ddof=1) / np.sqrt(len(d)):+.4f}, {d.mean() + 2.365 * d.std(ddof=1) / np.sqrt(len(d)):+.4f}]",
                 "seeds better than naive": int((d > 0).sum()), "max weight": round(part["max_weight"].mean(), 2)})
table = pl.DataFrame(rows)
with pl.Config(tbl_width_chars=200, tbl_cols=10):
    print(table)
print(f"n seeds = {N_SEEDS}; random order {random_ndcg:.4f}; logging policy order {logging_ndcg:.4f}; real-label reference {ref_ndcg:.4f}")

## Bias and variance across clipping thresholds

In [ ]:
means = [res.filter(pl.col("clip") == c)["ndcg_true"].mean() for c in CLIPS]
sds = [res.filter(pl.col("clip") == c)["ndcg_true"].std() for c in CLIPS]
labels_x = ["1\n(naive)", "2", "5", "10", "20", "none"]
fig, ax = plt.subplots(figsize=(6, 3.3))
ax.errorbar(range(len(CLIPS)), means, yerr=sds, fmt="o-", capsize=4, label="LightGBM on simulated clicks (mean and sd over seeds)")
ax.axhline(random_ndcg, color="grey", ls=":", label="random order")
ax.axhline(logging_ndcg, color="tab:red", ls="--", label="logging policy (popularity)")
ax.axhline(ref_ndcg, color="tab:green", ls="-.", label="trained on real labels (reference)")
ax.set_xticks(range(len(CLIPS)), labels_x)
ax.set_xlabel("clipping threshold M of the IPW weights"); ax.set_ylabel("NDCG@10 against true relevance")
ax.legend(fontsize=6); ax.set_title("Position-bias correction: bias vs variance")
plt.tight_layout(); plt.show()

# variance of the weights grows with M: the bias-variance trade-off in numbers
print(res.group_by("clip").agg(pl.col("weight_sd").mean().alias("sd_of_weights"), pl.col("max_weight").mean().alias("max_weight")).sort("clip"))

## Result

NDCG@10 against **true relevance** on 10,000 test queries; mean over 8 simulation seeds (each seed re-draws the clicks). References: random order 0.4418, the logging policy's popularity order 0.4666, a model trained on the real labels of all candidates 0.6207 (about 10 times more positives, so not a fair upper bound).

| Clipping M | NDCG@10 (true) | sd over seeds | difference to naive | 95% CI of the difference | seeds better than naive |
|---|---|---|---|---|---|
| 1 (naive, weights all 1) | 0.5258 | 0.0166 | - | - | - |
| 2 | 0.5276 | 0.0203 | +0.0018 | [-0.0052, +0.0089] | 4 of 8 |
| 5 | 0.5252 | 0.0205 | -0.0006 | [-0.0090, +0.0079] | 4 of 8 |
| 10 | 0.5179 | 0.0209 | -0.0079 | [-0.0159, +0.0002] | 2 of 8 |
| 20 | 0.5073 | 0.0171 | **-0.0185** | [-0.0321, -0.0049] | 1 of 8 |
| none | 0.5073 | 0.0171 | -0.0185 | [-0.0321, -0.0049] | 1 of 8 |

(With 20 positions the largest possible weight is 1/theta_20 = 20, so 'none' and 'M = 20' are the same.)

**Did IPW recover the true ordering better than naive? No, not in this experiment.**
- The naive model, trained on biased clicks, already ranks well above the logging policy (0.526 vs 0.467) and above random (0.442). The position bias did not cost it much here.
- Mild clipping (M = 2 or 5) is indistinguishable from naive: the differences are +0.0018 and -0.0006 with intervals that contain zero.
- Heavier weights **hurt**: plain IPW is 0.0185 worse than naive (interval excludes zero, only 1 of 8 seeds better).

**Why: the bias-variance trade-off, seen directly.** IPW removes the position bias in expectation, but it multiplies the few clicks at deep positions by up to 20. Each seed has only about 3,500 clicks, so a handful of heavily weighted clicks steer the model, and the weights' standard deviation rises from 0 (naive) to 0.05, 0.18, 0.32 and 0.44 as M grows. The added variance outweighs the removed bias. Clipping at M = 2 buys a little of the correction cheaply, and going higher trades away accuracy. This matches the usual advice to clip and self-normalise, but here even that did not produce a gain.

**Limits, and what was not tried.** Clicks are rare (0.43 percent), which is the main reason the correction cannot pay off; the position bias in this simulation is strong (propensity 1/k) but the logging policy's ordering happens to be positively related to true relevance, so the naive model is partly *right for the wrong reason*. A quick check with 10 times more clicks per seed, a logging policy that is unrelated or opposed to relevance, or more seeds was not run and might show a clear IPW gain. The result is for one simple click model with known propensities and LightGBM; it should not be read as 'IPW does not work'.

**Decision for the main pipeline.** The real OTTO labels are trained unweighted. The weight column exists in the training code and defaults to 1 (LightGBM `weight` and the neural loss), so IPW can be switched on if real positions ever exist.

## What I learned

To be written by the owner of the project.